# 03 · Muscle and fitness baseline

**Question:** before flight, where does each crew member sit for their age and sex? In flight, how far have they dropped from their own preflight value?

**Data:** *Body performance* dataset: 13,393 adults aged 21–64 (Korea Sports Promotion Foundation, via Kaggle). General human data, not astronauts.

**Method:** age- and sex-specific percentile curves (Gaussian-kernel weighted quantiles). The smoothing width is picked by 5-fold cross-validation, then checked on a 20% hold-out set.

**Why it matters in space:** astronauts can lose up to 20% of muscle mass on flights of 5–11 days ([CSA](https://asc-csa.gc.ca/eng/sciences/osm/muscles.asp)).

Steps: load → clean → fit → evaluate → save → export JSON for the backend.

## 0. Setup
**In Colab:** run the next cell and upload `bodyPerformance.csv` and `fitness.py` when asked.
**Locally (in the repo's `ml/` folder):** skip it, because the files are already there.

In [ ]:
# Colab only
import os, sys
if "google.colab" in sys.modules:
    from google.colab import files
    os.makedirs("data/raw", exist_ok=True); os.makedirs("src/crewhealth_ml", exist_ok=True)
    up = files.upload()  # choose bodyPerformance.csv and fitness.py
    for name, data in up.items():
        dest = f"data/raw/{name}" if name.endswith(".csv") else f"src/crewhealth_ml/{name}"
        open(dest, "wb").write(data)
    open("src/crewhealth_ml/__init__.py", "a").close()

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path("src").resolve()))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from crewhealth_ml import fitness as f
f.RAW = Path("data/raw/bodyPerformance.csv").resolve()
f.OUT = Path("outputs").resolve(); f.MODELS = Path("models").resolve()

## 1. Load and clean

In [ ]:
df, clean_report = f.load_clean(f.RAW)
print(json.dumps(clean_report, indent=1))
df[["age", "sex", "grip_force", "broad_jump", "sit_ups"]].describe().round(1)

Cleaning choices:
- 1 duplicate row removed.
- 0 kg grip and 0 cm jump treated as missing tests.
- 0 sit-ups kept, because nearly all are people aged 50+, so plausible.

## 2. Evaluate: pick the smoothing width by CV, then test on unseen people

In [ ]:
ev = f.evaluate(df)
rows = []
for tid, r in ev.items():
    rows.append({"test": tid, "bandwidth (years)": r["chosen_bandwidth_years"],
                 "coverage error (test)": r["coverage_error_on_test"],
                 "better than sex-only baseline (%)": r["improvement_pct"]})
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
qs = [q * 100 for q in f.QUANTILES]
for tid, r in ev.items():
    ax.plot(qs, [v * 100 for v in r["coverage_on_test"].values()], "o-", label=tid)
ax.plot([0, 100], [0, 100], "k--", lw=1, label="perfect")
ax.set_xlabel("percentile line"); ax.set_ylabel("% of unseen people below it")
ax.set_title("Calibration on the 20% hold-out"); ax.legend(); plt.show()

Points on the dashed line mean the percentiles are honest: for example, 25% of new people really fall below the 25th-percentile line.

## 3. Fit the final curves on all data and look at them

In [ ]:
models = f.fit_curves(df, {t: r["chosen_bandwidth_years"] for t, r in ev.items()})
ages = np.arange(f.AGE_MIN, f.AGE_MAX + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tid in zip(axes, f.TESTS):
    for sex, color in (("male", "tab:blue"), ("female", "tab:red")):
        q = f.predict_quantiles(models, tid, sex, ages)
        d = df[(df.sex == sex) & df[tid].notna()].sample(800, random_state=0)
        ax.scatter(d.age, d[tid], s=3, alpha=.15, color=color)
        ax.fill_between(ages, q[:, 0], q[:, -1], alpha=.15, color=color)
        ax.plot(ages, q[:, 3], color=color, label=f"{sex} median")
    ax.set_title(f.TESTS[tid][1]); ax.set_xlabel("age"); ax.set_ylabel(f.TESTS[tid][2])
axes[0].legend(); plt.suptitle("Median and 5th–95th percentile band by age and sex"); plt.show()

## 4. Personal baseline: synthetic demo crew in flight

In [ ]:
demo = f.demo_scores(models)
for c in demo["crew"]:
    print(c["crewId"], {k: f"p{v['percentile_for_age_sex']}" for k, v in c["preflight"].items()})
colors = {"normal": "tab:green", "watch": "orange", "alert": "tab:red"}
fig, ax = plt.subplots(figsize=(8, 4))
for c in demo["crew"]:
    w = c["inflight_grip_weekly"]
    ax.plot([x["week"] for x in w], [x["pct_change"] for x in w], "-", color="grey", lw=1)
    ax.scatter([x["week"] for x in w], [x["pct_change"] for x in w],
               c=[colors[x["status"]] for x in w], zorder=3)
    ax.annotate(c["crewId"], (w[-1]["week"], w[-1]["pct_change"]), xytext=(5, 0), textcoords="offset points")
ax.axhline(-f.RULES["suggested_watch_drop_pct"], color="orange", ls="--", lw=1, label="watch (team choice)")
ax.axhline(-f.RULES["suggested_alert_drop_pct"], color="tab:red", ls="--", lw=1, label="alert (CSA: up to 20% loss)")
ax.set_xlabel("week in flight"); ax.set_ylabel("grip change from own preflight (%)")
ax.set_title("Grip strength vs each person's own preflight value (synthetic crew)"); ax.legend(); plt.show()

Only grip strength can be measured in microgravity, with a hand dynamometer. Broad jump and sit-ups are preflight and after-landing tests (Earth, Moon or Mars surface).

## 5. Save the model and export JSON for the backend

In [ ]:
f.main()
print(json.dumps(json.load(open("outputs/fitness_norms.json"))["tests"]["grip_force"]["by_sex"]["female"]["38"]))

Outputs:
- `outputs/fitness_norms.json`: the percentile table the Java backend reads.
- `outputs/fitness_eval.json`: the cleaning and evaluation numbers.
- `outputs/fitness_demo_scores.json`: example scores for the synthetic crew.
- `models/fitness_quantiles.joblib`: the saved model.

**Limits:** Korean adults aged 21–64 on Earth, not astronauts. The 10% watch line is a team choice. Percentiles describe a person but don't judge them; only the change from their own baseline drives status.